# **Algorithmic Reasoning Distillation: Unsloth FastLanguageModel QLoRA Pipeline**
### **IT488 Course Project: Midsemester Evaluation (Phases 1 & 2 - 40% Scope)**
**Author:** Pranav Bhat P (Roll No: 231IT049)  
**Course:** IT488  
**Evaluation Target:** Midsemester Evaluation  
**Pipeline:** Pipeline B — Unsloth FastLanguageModel (Triton-Accelerated 4-bit NF4 Quantization)

---
### **📌 Pipeline Architecture**
1. **Dataset Engine (Phase 1)**: 100 competitive programming problems from **TACO (`BAAI/TACO`)** (800–1200 rating) across 10 algorithmic categories.
2. **Google Gemini Teacher Supervision**: High-fidelity structured reasoning rationales from `gemini-3.5-flash-lite`.
3. **Rationale-Consistency Filter (Phase 2)**: Semantic paradigm alignment validation.
4. **Stage 1 QLoRA Distillation**: Unsloth `FastLanguageModel` with Triton-kernel acceleration on `Qwen/Qwen2.5-0.5B`.
5. **Subprocess Sandbox Judge**: Timeout (2.0s) & memory (512MB) isolated execution judge.
6. **Empirical Evaluation Dashboard**: Top-1 and Top-3 accuracy, Macro F1, VRAM profiling, and Live Interactive Solver.

## **Step 1: Install Dependencies & Setup Environment**
Installs `unsloth`, `transformers`, `peft`, `accelerate`, and `bitsandbytes` optimized for Google Colab GPU runtimes.

In [1]:
# Install Unsloth accelerated fine-tuning stack
!pip install -q transformers peft accelerate bitsandbytes datasets scikit-learn rich seaborn matplotlib google-genai
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" || pip install -q unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 131.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 114.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 120.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 126.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.6 MB/s eta 0:0

In [2]:
# Recommended Unsloth import structure
try:
    import unsloth
    from unsloth import FastLanguageModel
except ImportError:
    pass

import os, sys, time, json, re, math, random, tempfile, subprocess, resource, gc
from dataclasses import dataclass, field, asdict
from enum import Enum
from typing import List, Dict, Any, Optional, Tuple, Union
from collections import defaultdict

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import Image, display

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"bfloat16 Supported: {torch.cuda.is_bf16_supported()}")
else:
    print("Running on CPU (Tip: In Colab, select Runtime -> Change runtime type -> T4 GPU)")

def get_gpu_memory_mb():
    if torch.cuda.is_available():
        return torch.cuda.max_memory_allocated() / (1024 * 1024)
    return 0.0

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch Version: 2.11.0+cu128
CUDA Available: True
Active GPU: Tesla T4
VRAM Available: 14.56 GB
bfloat16 Supported: False


## **Step 2: Load 100-Problem TACO Dataset from JSON**
Upload `taco_100_distillation_dataset.json` (or `taco_100_problems.json` + `teacher_traces_100.json`) to Colab's `/content/` directory.
The code loads the 100 TACO benchmark problems and Gemini teacher reasoning traces across 10 categories.

In [3]:
TARGET_TAG_TAXONOMY = [
    "dynamic programming",
    "greedy",
    "graphs",
    "math",
    "data structures",
    "trees",
    "brute force",
    "strings",
    "number theory",
    "binary search",
]
TAG_TO_IDX = {tag: i for i, tag in enumerate(TARGET_TAG_TAXONOMY)}
IDX_TO_TAG = {i: tag for i, tag in enumerate(TARGET_TAG_TAXONOMY)}

@dataclass
class DatasetItem:
    problem_id: str
    title: str
    ground_truth_tag: str
    rating: int
    statement: str
    input_spec: str
    output_spec: str
    teacher_rationale: str
    teacher_solution: str
    teacher_stated_tag: str
    sample_tests: List[Dict[str, str]]
    time_limit_ms: int = 2000
    memory_limit_mb: int = 512
    is_consistency_verified: bool = False

candidate_paths = [
    "taco_100_distillation_dataset.json",
    "data/taco_100_distillation_dataset.json",
    "/content/taco_100_distillation_dataset.json",
]

dataset_file = next((p for p in candidate_paths if os.path.exists(p)), None)
raw_data = None

# Auto-merge fallback if user uploaded taco_100_problems.json and teacher_traces_100.json separately
if dataset_file is None:
    prob_candidates = ["taco_100_problems.json", "data/taco_100_problems.json", "/content/taco_100_problems.json"]
    trace_candidates = ["teacher_traces_100.json", "data/teacher_traces_100.json", "/content/teacher_traces_100.json"]
    p_file = next((p for p in prob_candidates if os.path.exists(p)), None)
    t_file = next((p for p in trace_candidates if os.path.exists(p)), None)
    if p_file and t_file:
        print(f"Detected separate files: {p_file} and {t_file}. Auto-merging...")
        with open(p_file, 'r', encoding='utf-8') as f: p_list = json.load(f)
        with open(t_file, 'r', encoding='utf-8') as f: t_list = json.load(f)
        t_map = {t['problem_id']: t for t in t_list}
        raw_data = []
        for prob in p_list:
            t = t_map.get(prob['problem_id'], {})
            raw_data.append({
                **prob,
                'teacher_rationale': t.get('rationale', ''),
                'teacher_solution': t.get('solution_code', prob.get('raw_solution', '')),
                'teacher_stated_tag': t.get('stated_algorithmic_strategy', prob['ground_truth_tag'])
            })

if dataset_file is None and raw_data is None:
    print("Dataset JSON not found in directory.")
    try:
        from google.colab import files
        print("Please upload 'taco_100_distillation_dataset.json' below:")
        uploaded = files.upload()
        dataset_file = list(uploaded.keys())[0]
    except Exception as e:
        raise FileNotFoundError("Please upload 'taco_100_distillation_dataset.json' to Colab!")

if raw_data is None and dataset_file is not None:
    print(f"Loading dataset from: {dataset_file}")
    with open(dataset_file, "r", encoding="utf-8") as f:
        raw_data = json.load(f)
    if isinstance(raw_data, dict) and 'data' in raw_data:
        raw_data = raw_data['data']
    if isinstance(raw_data, dict) and 'problems' in raw_data:
        raw_data = raw_data['problems']

dataset = []
for d in raw_data:
    dataset.append(DatasetItem(
        problem_id=d["problem_id"],
        title=d.get("title", "Problem"),
        ground_truth_tag=d["ground_truth_tag"],
        rating=d.get("rating", 1000),
        statement=d["statement"],
        input_spec=d.get("input_spec", "Standard competitive programming input format."),
        output_spec=d.get("output_spec", "Standard competitive programming output format."),
        teacher_rationale=d.get("teacher_rationale", f"Step-by-step logic for {d['ground_truth_tag']}."),
        teacher_solution=d.get("teacher_solution", d.get("raw_solution", "")),
        teacher_stated_tag=d.get("teacher_stated_tag", d["ground_truth_tag"]),
        sample_tests=d.get("sample_tests", []),
        time_limit_ms=d.get("time_limit_ms", 2000),
        memory_limit_mb=d.get("memory_limit_mb", 512)
    ))

print(f"✓ Successfully loaded {len(dataset)} problems across 10 algorithmic categories:\n")
for i, tag in enumerate(TARGET_TAG_TAXONOMY, start=1):
    count = sum(1 for item in dataset if item.ground_truth_tag == tag)
    print(f"  {i:2d}. {tag.title():<22} : {count} problems")

Loading dataset from: taco_100_distillation_dataset.json
✓ Successfully loaded 100 problems across 10 algorithmic categories:

   1. Dynamic Programming    : 10 problems
   2. Greedy                 : 10 problems
   3. Graphs                 : 10 problems
   4. Math                   : 10 problems
   5. Data Structures        : 10 problems
   6. Trees                  : 10 problems
   7. Brute Force            : 10 problems
   8. Strings                : 10 problems
   9. Number Theory          : 10 problems
  10. Binary Search          : 10 problems


## **Step 3: Sandboxed Subprocess Test Judge (Phase 1 Deliverable)**
Enforces strict timeouts (2.0s) and memory caps (512MB) in isolated subprocesses.

In [4]:
class Verdict(str, Enum):
    AC = "AC"    # Accepted
    WA = "WA"    # Wrong Answer
    TLE = "TLE"  # Time Limit Exceeded
    MLE = "MLE"  # Memory Limit Exceeded
    RE = "RE"    # Runtime Error
    CE = "CE"    # Compilation / Syntax Error

@dataclass
class TestResult:
    test_index: int
    input_data: str
    expected_output: str
    actual_output: str
    verdict: Verdict
    runtime_ms: float
    error_message: Optional[str] = None

@dataclass
class ExecutionResult:
    verdict: Verdict
    tests_passed: int
    total_tests: int
    pass_rate: float
    max_runtime_ms: float
    test_results: List[TestResult] = field(default_factory=list)

class SandboxJudge:
    def __init__(self, default_timeout_s: float = 2.0, max_memory_mb: int = 512):
        self.default_timeout_s = default_timeout_s
        self.max_memory_mb = max_memory_mb

    @staticmethod
    def _normalize_output(text: str) -> List[str]:
        if not text:
            return []
        lines = text.strip().replace("\r\n", "\n").split("\n")
        return [line.rstrip() for line in lines if line.rstrip()]

    def _set_limits(self, memory_mb: int) -> None:
        try:
            mem_bytes = memory_mb * 1024 * 1024
            resource.setrlimit(resource.RLIMIT_AS, (mem_bytes, mem_bytes))
        except (ValueError, OSError):
            pass

    def evaluate_python_solution(self, code: str, test_cases: List[Dict[str, str]], timeout_s: Optional[float] = None) -> ExecutionResult:
        timeout = timeout_s or self.default_timeout_s
        if not test_cases:
            return ExecutionResult(Verdict.AC, 0, 0, 1.0, 0.0)

        try:
            compile(code, "<string>", "exec")
        except SyntaxError as e:
            return ExecutionResult(Verdict.CE, 0, len(test_cases), 0.0, 0.0)

        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
            f.write(code)
            script_path = f.name

        try:
            results = []
            max_ms = 0.0
            overall_verdict = Verdict.AC
            for idx, tc in enumerate(test_cases):
                inp = tc.get("input_data", "")
                expected = tc.get("output_data", "")
                start_t = time.perf_counter()
                try:
                    proc = subprocess.run(
                        [sys.executable, script_path],
                        input=inp,
                        text=True,
                        capture_output=True,
                        timeout=timeout,
                        preexec_fn=lambda: self._set_limits(self.max_memory_mb)
                    )
                    runtime_ms = (time.perf_counter() - start_t) * 1000.0
                    max_ms = max(max_ms, runtime_ms)
                    if proc.returncode != 0:
                        verdict = Verdict.RE
                        if overall_verdict == Verdict.AC: overall_verdict = Verdict.RE
                        results.append(TestResult(idx, inp, expected, proc.stdout, verdict, runtime_ms, proc.stderr[:200]))
                        continue
                    norm_exp = self._normalize_output(expected)
                    norm_act = self._normalize_output(proc.stdout)
                    verdict = Verdict.AC if norm_exp == norm_act else Verdict.WA
                    if verdict != Verdict.AC and overall_verdict == Verdict.AC:
                        overall_verdict = Verdict.WA
                    results.append(TestResult(idx, inp, expected, proc.stdout, verdict, runtime_ms))
                except subprocess.TimeoutExpired:
                    runtime_ms = timeout * 1000.0
                    max_ms = max(max_ms, runtime_ms)
                    if overall_verdict == Verdict.AC: overall_verdict = Verdict.TLE
                    results.append(TestResult(idx, inp, expected, "", Verdict.TLE, runtime_ms))
            passed = sum(1 for r in results if r.verdict == Verdict.AC)
            return ExecutionResult(overall_verdict, passed, len(test_cases), passed / len(test_cases), max_ms, results)
        finally:
            if os.path.exists(script_path):
                os.remove(script_path)

judge = SandboxJudge()
print("=== Testing Sandboxed Judge on Sample Solutions ===")
for item in dataset[:5]:
    res = judge.evaluate_python_solution(item.teacher_solution, item.sample_tests)
    print(f"Problem {item.problem_id} ({item.ground_truth_tag}): Verdict = {res.verdict.value}, Passed = {res.tests_passed}/{res.total_tests}, Max Runtime = {res.max_runtime_ms:.1f}ms")

=== Testing Sandboxed Judge on Sample Solutions ===
Problem TACO-DY-01 (dynamic programming): Verdict = WA, Passed = 1/3, Max Runtime = 222.3ms
Problem TACO-DY-02 (dynamic programming): Verdict = WA, Passed = 0/3, Max Runtime = 2000.0ms
Problem TACO-DY-03 (dynamic programming): Verdict = AC, Passed = 3/3, Max Runtime = 89.1ms
Problem TACO-DY-04 (dynamic programming): Verdict = RE, Passed = 0/2, Max Runtime = 86.2ms
Problem TACO-DY-05 (dynamic programming): Verdict = RE, Passed = 0/3, Max Runtime = 88.2ms


## **Step 4: Rationale-Consistency Semantic Filter (Phase 2 Deliverable)**
Purifies the teacher distillation dataset before model training.

In [5]:
PARADIGM_SIGNATURES = {
    "dynamic programming": ["dynamic programming", "dp", "memoization", "subproblem", "state transition", "optimal substructure", "knapsack"],
    "greedy": ["greedy", "locally optimal", "greedy choice", "sort and choose", "exchange argument", "greedy allocation"],
    "graphs": ["graph", "vertex", "vertices", "edge", "edges", "dfs", "bfs", "connected component", "shortest path", "traversal"],
    "math": ["math", "mathematical", "formula", "parity", "even", "odd", "combinatorics", "equation"],
    "data structures": ["data structure", "heap", "priority queue", "hash map", "hash table", "stack", "queue", "dsu", "frequency array"],
    "trees": ["tree", "root", "leaf", "ancestor", "subtree", "depth", "forest", "hierarchy"],
    "brute force": ["brute force", "simulate", "simulation", "iterate", "try all", "exhaustive", "scan each"],
    "strings": ["string", "character", "characters", "substring", "prefix", "suffix", "vowel", "consonant", "lexicographical"],
    "number theory": ["number theory", "prime", "primes", "divisor", "divisors", "gcd", "lcm", "modulo", "modular", "sieve"],
    "binary search": ["binary search", "search space", "monotonic", "bisect", "upper_bound", "lower_bound", "logarithmic"],
}

class FilterVerdict(str, Enum):
    RETAINED = "RETAINED"
    REJECTED_TAG_MISMATCH = "REJECTED_TAG_MISMATCH"
    REJECTED_INSUFFICIENT_REASONING = "REJECTED_INSUFFICIENT_REASONING"
    REJECTED_CONTRADICTORY_LOGIC = "REJECTED_CONTRADICTORY_LOGIC"

@dataclass
class FilterResult:
    problem_id: str
    ground_truth_tag: str
    verdict: FilterVerdict
    is_valid: bool
    consistency_score: float
    reasons: List[str]

class RationaleFilter:
    def __init__(self, min_length: int = 40, threshold: float = 0.5):
        self.min_length = min_length
        self.threshold = threshold

    def evaluate_item(self, item: DatasetItem) -> FilterResult:
        gt_tag = item.ground_truth_tag.lower().strip()
        stated_tag = item.teacher_stated_tag.lower().strip() if item.teacher_stated_tag else ""
        rationale = item.teacher_rationale.lower()

        if stated_tag and stated_tag != gt_tag:
            for other_tag in PARADIGM_SIGNATURES:
                if other_tag != gt_tag and other_tag in stated_tag:
                    return FilterResult(item.problem_id, gt_tag, FilterVerdict.REJECTED_TAG_MISMATCH, False, 0.0, [f"Contradictory tag '{stated_tag}'"])

        if len(rationale.split()) < 8 or len(rationale) < self.min_length:
            return FilterResult(item.problem_id, gt_tag, FilterVerdict.REJECTED_INSUFFICIENT_REASONING, False, 0.1, ["Insufficient length"])

        target_kws = PARADIGM_SIGNATURES.get(gt_tag, [gt_tag])
        matches = [kw for kw in target_kws if re.search(r'\b' + re.escape(kw) + r'\b', rationale)]

        score = min(1.0, 0.4 + 0.2 * len(matches)) if matches else (0.6 if gt_tag in rationale else 0.2)
        verdict = FilterVerdict.RETAINED if score >= self.threshold else FilterVerdict.REJECTED_INSUFFICIENT_REASONING
        return FilterResult(item.problem_id, gt_tag, verdict, verdict == FilterVerdict.RETAINED, score, matches)

    def filter_dataset(self, dataset: List[DatasetItem]):
        retained, discarded = [], []
        for item in dataset:
            res = self.evaluate_item(item)
            if res.is_valid:
                item.is_consistency_verified = True
                retained.append(item)
            else:
                discarded.append(item)
        return retained, discarded

filt = RationaleFilter()
retained, discarded = filt.filter_dataset(dataset)
print(f"Rationale-Consistency Filter: Retained = {len(retained)}/{len(dataset)} ({len(retained)/len(dataset)*100:.1f}%), Discarded = {len(discarded)}")

# Dataset Partitioning (70% Train, 10% Val, 20% Test)
random.seed(42)
tag_groups = defaultdict(list)
for item in retained:
    tag_groups[item.ground_truth_tag].append(item)

train_split, val_split, test_split = [], [], []
for tag, items in tag_groups.items():
    shuffled = list(items)
    random.shuffle(shuffled)
    n = len(shuffled)
    n_train = max(1, int(n * 0.7))
    n_val = max(1, int(n * 0.1))
    train_split.extend(shuffled[:n_train])
    val_split.extend(shuffled[n_train:n_train + n_val])
    test_split.extend(shuffled[n_train + n_val:])

print(f"Distillation Dataset Partition: Train = {len(train_split)}, Val = {len(val_split)}, Test = {len(test_split)}")

class DistillationProblemDataset(Dataset):
    def __init__(self, items: List[DatasetItem], tokenizer, max_length: int = 256):
        self.items = items
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        text = f"Problem: {item.title}\n{item.statement}\nInput: {item.input_spec}"
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(TAG_TO_IDX[item.ground_truth_tag], dtype=torch.long)
        }

Rationale-Consistency Filter: Retained = 81/100 (81.0%), Discarded = 19
Distillation Dataset Partition: Train = 55, Val = 9, Test = 17


## **Step 5: Unsloth FastLanguageModel QLoRA Fine-Tuning**
High-performance fine-tuning pipeline utilizing **Unsloth's custom Triton kernels** and zero-overhead LoRA modules on `Qwen/Qwen2.5-0.5B`.

In [6]:
STUDENT_MODEL_ID = "Qwen/Qwen2.5-0.5B"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"\n[Unsloth Pipeline] Initializing FastLanguageModel on {STUDENT_MODEL_ID}...")
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# Sequence classification wrapper for Unsloth FastLanguageModel
class UnslothClassifierModel(nn.Module):
    def __init__(self, unsloth_model, num_labels, feature_dim=151936):
        super().__init__()
        self.unsloth_model = unsloth_model
        dev = next(unsloth_model.parameters()).device
        self.score = nn.Linear(feature_dim, num_labels, bias=False).to(device=dev, dtype=torch.float32)

    def forward(self, input_ids, attention_mask, labels=None):
        # Forward through Unsloth's Triton-accelerated transformer WITHOUT passing labels
        outputs = self.unsloth_model(input_ids=input_ids, attention_mask=attention_mask)
        logits_or_hidden = outputs.logits if hasattr(outputs, "logits") else outputs[0]
        seq_lens = attention_mask.sum(dim=1) - 1
        batch_idx = torch.arange(input_ids.shape[0], device=input_ids.device)
        pooled = logits_or_hidden[batch_idx, seq_lens].to(torch.float32)

        if pooled.shape[-1] != self.score.in_features:
            self.score = nn.Linear(pooled.shape[-1], self.score.out_features, bias=False).to(device=pooled.device, dtype=torch.float32)

        logits = self.score(pooled)
        loss = None
        if labels is not None:
            loss = nn.functional.cross_entropy(logits, labels)
        return type("ClassificationOutput", (), {"loss": loss, "logits": logits})

# Load Model with Unsloth Triton Kernels
model_unsloth_base, tokenizer_unsloth = FastLanguageModel.from_pretrained(
    model_name=STUDENT_MODEL_ID,
    max_seq_length=256,
    dtype=None,
    load_in_4bit=True,
)

peft_unsloth = FastLanguageModel.get_peft_model(
    model_unsloth_base,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

vocab_size = getattr(model_unsloth_base.config, "vocab_size", 151936)
model_unsloth = UnslothClassifierModel(
    peft_unsloth,
    num_labels=len(TARGET_TAG_TAXONOMY),
    feature_dim=vocab_size
)
print("✓ Successfully initialized Unsloth FastLanguageModel with optimized Triton kernels!")

# Train Pipeline B (Unsloth)
train_dataset_unsloth = DistillationProblemDataset(train_split, tokenizer_unsloth)
train_loader_unsloth = DataLoader(train_dataset_unsloth, batch_size=4, shuffle=True)
optimizer_unsloth = torch.optim.AdamW(model_unsloth.parameters(), lr=2e-4, weight_decay=0.01)
num_epochs = 4

unsloth_train_losses = []
start_t_unsloth = time.perf_counter()
model_unsloth.train()

for epoch in range(1, num_epochs + 1):
    epoch_loss = 0.0
    for batch in train_loader_unsloth:
        optimizer_unsloth.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model_unsloth(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer_unsloth.step()
        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader_unsloth)
    unsloth_train_losses.append(avg_loss)
    print(f"  [Unsloth] Epoch {epoch}/{num_epochs} Loss: {avg_loss:.4f}")

unsloth_total_time = time.perf_counter() - start_t_unsloth
unsloth_peak_vram_mb = get_gpu_memory_mb()
print(f"✓ [Unsloth] Training Finished in {unsloth_total_time:.2f}s | Peak VRAM: {unsloth_peak_vram_mb:.1f} MB")


[Unsloth Pipeline] Initializing FastLanguageModel on Qwen/Qwen2.5-0.5B...
==((====))==  Unsloth 2026.8.22: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Not an error, but Unsloth cannot patch MLP layers with our manual autograd engine since either LoRA adapters
are not enabled or a bias term (like in Qwen) is used.
Unsloth 2026.8.22 patched 24 layers with 24 QKV layers, 24 O layers and 0 MLP layers.
`use_return_dict` is deprecated! Use `return_dict` instead!


✓ Successfully initialized Unsloth FastLanguageModel with optimized Triton kernels!
  [Unsloth] Epoch 1/4 Loss: 59.8709
  [Unsloth] Epoch 2/4 Loss: 23.7168
  [Unsloth] Epoch 3/4 Loss: 4.4034
  [Unsloth] Epoch 4/4 Loss: 3.0124
✓ [Unsloth] Training Finished in 21.55s | Peak VRAM: 971.1 MB


## **Step 6: Empirical Evaluation & Performance Metrics**
Evaluates the Unsloth model on the unseen test partition and visualizes classification accuracy and convergence:

In [7]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def evaluate_model(model, tokenizer, test_items):
    model.eval()
    test_dataset = DistillationProblemDataset(test_items, tokenizer)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)
    y_true = [item.ground_truth_tag for item in test_items]
    y_pred = []
    y_probs = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
            pred_idx = int(np.argmax(probs))
            y_pred.append(IDX_TO_TAG[pred_idx])
            y_probs.append(probs)

    acc = accuracy_score(y_true, y_pred)
    top_3_correct = 0
    for i, true_tag in enumerate(y_true):
        top_3_indices = np.argsort(y_probs[i])[-3:]
        top_3_tags = [IDX_TO_TAG[idx] for idx in top_3_indices]
        if true_tag in top_3_tags:
            top_3_correct += 1
    top_3_acc = top_3_correct / len(y_true)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, labels=TARGET_TAG_TAXONOMY, average="macro", zero_division=0)
    return acc, top_3_acc, f1, y_pred, y_probs

unsloth_acc, unsloth_top3, unsloth_f1, _, _ = evaluate_model(model_unsloth, tokenizer_unsloth, test_split)

print("\n=========================================================================")
print("  📊 EVALUATION BENCHMARK: UNSLOTH FASTLANGUAGEMODEL QLORA              ")
print("=========================================================================")
print(f"{'Metric':<35} | {'Value':<20}")
print("-" * 58)
print(f"{'Base Student LLM':<35} | {STUDENT_MODEL_ID:<20}")
print(f"{'Quantization Precision':<35} | {'4-bit NF4 (Triton)':<20}")
print(f"{'Trainable Parameters':<35} | {'2.17M (0.44%)':<20}")
print(f"{'Total Training Time':<35} | {f'{unsloth_total_time:.2f}s':<20}")
print(f"{'Peak VRAM Memory':<35} | {f'{unsloth_peak_vram_mb:.1f} MB':<20}")
print(f"{'Top-1 Classification Accuracy':<35} | {f'{unsloth_acc*100:.2f}%':<20}")
print(f"{'Top-3 Classification Accuracy':<35} | {f'{unsloth_top3*100:.2f}%':<20}")
print(f"{'Macro F1 Score':<35} | {f'{unsloth_f1:.4f}':<20}")
print("=" * 58)

# Plot Convergence Curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
epochs = list(range(1, num_epochs + 1))
axes[0].plot(epochs, unsloth_train_losses, marker="s", linewidth=2.5, color="#50E3C2", label="Unsloth Loss")
axes[0].set_xlabel("Epoch", fontweight="bold")
axes[0].set_ylabel("Cross-Entropy Loss", fontweight="bold")
axes[0].set_title("Training Loss Trajectory", pad=10, fontweight="bold")
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend()

acc_metrics = ["Top-1 Acc", "Top-3 Acc", "Macro F1"]
acc_vals = [unsloth_acc * 100, unsloth_top3 * 100, unsloth_f1 * 100]
axes[1].bar(acc_metrics, acc_vals, color=["#50E3C2", "#4A90E2", "#E94E77"], width=0.5)
axes[1].set_ylabel("Percentage (%)", fontweight="bold")
axes[1].set_title("Test Set Accuracy Metrics", pad=10, fontweight="bold")
for i, v in enumerate(acc_vals):
    axes[1].text(i, v + 2, f"{v:.1f}%", ha="center", fontweight="bold")
axes[1].grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


  📊 EVALUATION BENCHMARK: UNSLOTH FASTLANGUAGEMODEL QLORA              
Metric                              | Value               
----------------------------------------------------------
Base Student LLM                    | Qwen/Qwen2.5-0.5B   
Quantization Precision              | 4-bit NF4 (Triton)  
Trainable Parameters                | 2.17M (0.44%)       
Total Training Time                 | 21.55s              
Peak VRAM Memory                    | 971.1 MB            
Top-1 Classification Accuracy       | 5.88%               
Top-3 Classification Accuracy       | 23.53%              
Macro F1 Score                      | 0.0222              


## **Step 7: Interactive Live Problem Solver**
Run live predictions with the fine-tuned Unsloth model on custom competitive programming problem statements:

In [8]:
def predict_problem(problem_statement: str, top_k: int = 3):
    model_unsloth.eval()
    enc = tokenizer_unsloth(problem_statement, truncation=True, max_length=256, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model_unsloth(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    sorted_idx = np.argsort(probs)[::-1]
    return [(IDX_TO_TAG[idx], float(probs[idx])) for idx in sorted_idx[:top_k]]

demo_problem = """
Given a weighted undirected graph with n vertices and m edges. Find the shortest path
between vertex 1 and vertex n using Dijkstra's algorithm with a min-priority queue.
"""

preds = predict_problem(demo_problem)
print("=== LIVE INFERENCE PREDICTION (UNSLOTH) ===")
print(f"Problem Statement:\n{demo_problem.strip()}\n")
print(f"Predicted Algorithmic Paradigm: {preds[0][0].upper()} (Confidence: {preds[0][1]*100:.1f}%)\n")
print("Top-3 Confidence Distribution:")
for r, (tag, prob) in enumerate(preds, 1):
    print(f"  {r}. {tag.title():<22} [{('█'*int(prob*25)):<25}] {prob*100:.1f}%")

=== LIVE INFERENCE PREDICTION (UNSLOTH) ===
Problem Statement:
Given a weighted undirected graph with n vertices and m edges. Find the shortest path
between vertex 1 and vertex n using Dijkstra's algorithm with a min-priority queue.

Predicted Algorithmic Paradigm: TREES (Confidence: 100.0%)

Top-3 Confidence Distribution:
  1. Trees                  [████████████████████████ ] 100.0%
  2. Binary Search          [                         ] 0.0%
  3. Graphs                 [                         ] 0.0%


### **✓ Midsemester Evaluation Verification (40% Scope Complete)**
- **Phase 1 (20%)**: 100 TACO benchmark problems, Google Gemini teacher traces, and Subprocess Sandbox Judge.
- **Phase 2 (20%)**: Rationale-Consistency Filter, **Unsloth FastLanguageModel QLoRA Distillation** on `Qwen2.5-0.5B`, Evaluation Dashboard, and Live Inference.
- **Evaluation Target**: Ready for demonstration and evaluation.